In [1]:
import json

In [2]:
with open("profiles/spaceship_titanic_students/pipeline_1_1.json") as f:
    content = json.load(f)

In [3]:
content.keys()

dict_keys(['alloc_samples', 'args', 'async_profile', 'combined_stacks', 'combined_stacks_timeline', 'elapsed_time_sec', 'entrypoint_dir', 'filename', 'files', 'gpu', 'gpu_device', 'growth_rate', 'main_thread_id', 'max_footprint_fname', 'max_footprint_lineno', 'max_footprint_mb', 'max_footprint_python_fraction', 'memory', 'memory_stacks', 'native_allocations_mb', 'program', 'samples', 'stacks', 'start_time_absolute', 'start_time_perf'])

In [4]:
content["files"].keys()

dict_keys(['/Users/tue/Documents/10_Projects/deai-pro/spaceship_titanic_students/pipeline_1_1.py', '<exec@_internal.py:64>'])

In [ ]:
lines = content["files"][
    "/Users/tue/Documents/10_Projects/deai-pro/spaceship_titanic_students/pipeline_1_1.py"
]["lines"]


In [16]:
len(lines)

114

In [17]:
lines

[{'async_task_names': [],
  'cpu_samples_list': [],
  'end_function_line': 0,
  'end_outermost_loop': 1,
  'end_region_line': 1,
  'is_coroutine': False,
  'line': '"""Raw features; logistic C=0.1. Read passenger files from input/."""\n',
  'lineno': 1,
  'memory_samples': [],
  'n_async_await_percent': 0.0,
  'n_async_concurrency_mean': 0.0,
  'n_async_concurrency_peak': 0.0,
  'n_avg_mb': 0.0,
  'n_copy_mb_s': 0.0,
  'n_core_utilization': 0.0,
  'n_cpu_percent_c': 0.0,
  'n_cpu_percent_python': 0.0,
  'n_gpu_avg_memory_mb': 0.0,
  'n_gpu_peak_memory_mb': 0.0,
  'n_gpu_percent': 0,
  'n_growth_mb': 0.0,
  'n_malloc_mb': 0.0,
  'n_mallocs': 0,
  'n_peak_mb': 0.0,
  'n_python_fraction': 0,
  'n_sys_percent': 0.0,
  'n_usage_fraction': 0.0,
  'start_function_line': 0,
  'start_outermost_loop': 1,
  'start_region_line': 1},
 {'async_task_names': [],
  'cpu_samples_list': [],
  'end_function_line': 0,
  'end_outermost_loop': 2,
  'end_region_line': 2,
  'is_coroutine': False,
  'line': '\n

In [18]:
import_lines = [l for l in lines if "import " in l["line"]]

In [19]:
len(import_lines)

10

In [20]:
for l in import_lines:
    print(l)

{'async_task_names': [], 'cpu_samples_list': [], 'end_function_line': 0, 'end_outermost_loop': 3, 'end_region_line': 3, 'is_coroutine': False, 'line': 'from pathlib import Path\n', 'lineno': 3, 'memory_samples': [], 'n_async_await_percent': 0.0, 'n_async_concurrency_mean': 0.0, 'n_async_concurrency_peak': 0.0, 'n_avg_mb': 0.0, 'n_copy_mb_s': 0.0, 'n_core_utilization': 0.0, 'n_cpu_percent_c': 0.0, 'n_cpu_percent_python': 0.0, 'n_gpu_avg_memory_mb': 0.0, 'n_gpu_peak_memory_mb': 0.0, 'n_gpu_percent': 0, 'n_growth_mb': 0.0, 'n_malloc_mb': 0.0, 'n_mallocs': 0, 'n_peak_mb': 0.0, 'n_python_fraction': 0, 'n_sys_percent': 0.0, 'n_usage_fraction': 0.0, 'start_function_line': 0, 'start_outermost_loop': 3, 'start_region_line': 3}
{'async_task_names': [], 'cpu_samples_list': [], 'end_function_line': 0, 'end_outermost_loop': 4, 'end_region_line': 4, 'is_coroutine': False, 'line': 'import numpy as np\n', 'lineno': 4, 'memory_samples': [[38293833, 1.0000381469726562], [403453958, 2.0636606216430664], 

total_s and share_of_runtime: the two headline numbers. Seconds let you compare pipelines of different lengths. The share tells you how much a pipeline could gain at most by fixing this category.
python_s / native_s / system_s: these show what kind of fix would help.
High Python time means interpreter overhead (e.g. apply with a lambda), which vectorizing can fix.
High native time means the library is doing real work, so the only option is doing less of it.
High system time means waiting on disk or I/O, which is where your "writing to disk" category should show up.
core_utilization: the share of the machine's cores in use, weighted by time. A low value during heavy model training means work runs on one core and could be parallelized. That's relevant to your scheduling exercise.
malloc_mb and n_mallocs: how much memory gets allocated and freed again. Lots of allocation means temporary copies, which cost time even when peak memory stays low.
python_alloc_fraction: whether the allocations come from Python objects or native arrays. Python-heavy usually points to object columns or strings in pandas.
peak_mb: decides how many pipelines can run at the same time without running out of memory. Peaks can't be added up, so this takes the largest line peak.
copy_mb: data copied between Python and native code, e.g. pandas → numpy conversions. This is time spent without any computation.

In [21]:
def compute_total(category, lines, elapsed_time_sec):
    """Aggregate Scalene line records of one category into absolute totals.

    CPU percentages are shares of the sampled run time (~ elapsed_time_sec),
    so they are additive across lines and convert to seconds.
    """
    active = [
        l for l in lines
        if l["n_cpu_percent_python"] + l["n_cpu_percent_c"] + l["n_sys_percent"] > 0
        or l["n_malloc_mb"] > 0
    ]
    to_sec = elapsed_time_sec / 100
    python_s = sum(l["n_cpu_percent_python"] for l in active) * to_sec
    native_s = sum(l["n_cpu_percent_c"] for l in active) * to_sec
    system_s = sum(l["n_sys_percent"] for l in active) * to_sec
    total_s = python_s + native_s + system_s
    cpu_pct = sum(l["n_cpu_percent_python"] + l["n_cpu_percent_c"] for l in active)
    malloc_mb = sum(l["n_malloc_mb"] for l in active)
    return {
        "category": category,
        "n_relevant_lines": len(active),
        "total_s": total_s,
        "share_of_runtime": total_s / elapsed_time_sec if elapsed_time_sec else 0.0,
        "python_s": python_s,
        "native_s": native_s,
        "system_s": system_s,
        # Time-weighted: lines that ran longer count more.
        "core_utilization": (
            sum(l["n_core_utilization"] * (l["n_cpu_percent_python"] + l["n_cpu_percent_c"]) for l in active) / cpu_pct
            if cpu_pct else 0.0
        ),
        "malloc_mb": malloc_mb,
        "n_mallocs": sum(l["n_mallocs"] for l in active),
        # Allocation-weighted share of memory allocated by Python (vs. native) code.
        "python_alloc_fraction": (
            sum(l["n_python_fraction"] * l["n_malloc_mb"] for l in active) / malloc_mb
            if malloc_mb else 0.0
        ),
        # Peaks are not additive: the category's peak is the largest line peak.
        "peak_mb": max((l["n_peak_mb"] for l in active), default=0.0),
        "copy_mb": sum(l["n_copy_mb_s"] for l in active) * elapsed_time_sec,
    }


compute_total("import", import_lines, content["elapsed_time_sec"])

{'category': 'import',
 'n_relevant_lines': 4,
 'total_s': 1.4344518487034492,
 'share_of_runtime': 0.7125845219054082,
 'python_s': 0.289526472057876,
 'native_s': 0.7428236058008307,
 'system_s': 0.4021017708447426,
 'core_utilization': 0.07197252535284938,
 'malloc_mb': 73.66714000701904,
 'n_mallocs': 0,
 'python_alloc_fraction': 0.9757874440034544,
 'peak_mb': 45.250155448913574,
 'copy_mb': 85.76226425170898}

In [22]:
with open("libraries/spaceship_titanic_students/pipeline_1_1.json") as f:
    libraries = {int(lineno): modules for lineno, modules in json.load(f)["lines"].items()}

In [23]:
# Checked in order: a line that calls a model library is "models" even if it also touches pandas,
# e.g. classifier.fit(X.iloc[rows]). builtins carry no category.
CATEGORY_PREFIXES = {
    "models": (
        "sklearn.linear_model", "sklearn.ensemble",
        "sklearn.base",  # estimator .fit()
        "sklearn.model_selection",  # CV splitting
        "sklearn.utils._param_validation",  # wraps metrics such as accuracy_score
        "lightgbm", "xgboost", "catboost",
    ),
    "data handling": (
        "pandas", "numpy",
        "sklearn.compose", "sklearn.pipeline", "sklearn.impute", "sklearn.preprocessing",
        "sklearn.utils._set_output",  # wraps transformer fit_transform / transform
    ),
    "setup": ("importlib", "pathlib"),
}


def categorize_line(line, modules):
    for category, prefixes in CATEGORY_PREFIXES.items():
        if any(module.startswith(prefixes) for module in modules):
            return category
    # Untraced lines: imports of already-loaded modules, or plain Python inside helpers
    # such as split_cabin that pandas calls via apply.
    if line.lstrip().startswith(("import ", "from ")):
        return "setup"
    return "data handling"

In [24]:
from collections import defaultdict
import pandas as pd

lines_by_category = defaultdict(list)
for l in lines:
    lines_by_category[categorize_line(l["line"], libraries.get(l["lineno"], {}))].append(l)

totals = pd.DataFrame(
    [compute_total(category, category_lines, content["elapsed_time_sec"])
     for category, category_lines in lines_by_category.items()]
).set_index("category")
totals

,n_relevant_lines,total_s,share_of_runtime,python_s,native_s,system_s,core_utilization,malloc_mb,n_mallocs,python_alloc_fraction,peak_mb,copy_mb
category,,,,,,,,,,,,
data handling,19,0.165034,0.081983,0.020521,0.119632,0.024880,0.085748,59.703537,0,0.237566,13.892122,11.070651
setup,4,1.434452,0.712585,0.289526,0.742824,0.402102,0.071973,73.667140,0,0.975787,45.250155,85.762264
models,3,0.375070,0.186321,0.075201,0.274564,0.025304,0.093304,0.000000,0,0.000000,0.000000,0.000000
